In [ ]:

import pandas as pd
from pathlib import Path


DATA_DIR = Path('Data')

FX_FILES = {
    'AUD': 'DEXUSAL.csv',
    'EUR': 'DEXUSEU.csv',
    'GBP': 'DEXUSUK.csv',
    'CHF': 'DEXSZUS.csv',
    'JPY': 'EXJPUS.csv',
}

RATE_FILES = {
    'USD': 'USA.csv',
    'AUD': 'AUSTRALIA.csv',
    'EUR': 'EUROPE.csv',
    'JPY': 'JAPAN.csv',
    'CHF': 'SWITZERLAND.csv',
    'GBP': 'GREATBRITAIN.csv',
}

COUNTRY_CCY = {
    'AUSTRALIA': 'AUD',
    'FRANCE': 'EUR',
    'GERMANY': 'EUR',
    'JAPAN': 'JPY',
    'SWITZERLAND': 'CHF',
    'UNITED KINGDOM': 'GBP',
}


# 2. Load equity index returns (semicolon-separated)

raw = pd.read_csv(DATA_DIR / 'country_data.csv', sep=';')
raw['date'] = pd.to_datetime(raw['date'])

local_ret = (
    raw.pivot(index='date', columns='country', values='mportret')
       .sort_index()
       .astype(float)
)

# 3. Load FX data and invert where needed

fx = {}
for ccy, fname in FX_FILES.items():
    s = pd.read_csv(DATA_DIR / fname)
    date_col = 'DATE' if 'DATE' in s.columns else 'observation_date'
    s[date_col] = pd.to_datetime(s[date_col])
    value_col = [c for c in s.columns if c != date_col][0]

    s = (
        s.rename(columns={date_col: 'date', value_col: 'fx'})
         .set_index('date')['fx']
         .astype(float)
         .dropna()
         .resample('M').last()
    )

    if fname.upper().endswith('US.csv') and not fname.upper().startswith('DEXUS'):
        s = 1.0 / s
    if fname.upper().endswith('ZUS.csv'):
        s = 1.0 / s

    fx[ccy] = s

fx = pd.concat(fx, axis=1).sort_index()

# 4. Load 3-month risk-free rates and convert to 1-month simple returns

rf = {}
for ccy, fname in RATE_FILES.items():
    r = pd.read_csv(DATA_DIR / fname)
    date_col = 'DATE' if 'DATE' in r.columns else 'observation_date'
    r[date_col] = pd.to_datetime(r[date_col])
    value_col = [c for c in r.columns if c != date_col][0]

    r = (
        r.rename(columns={date_col: 'date', value_col: 'rf'})
         .set_index('date')['rf']
         .astype(float)
         .dropna()
         .resample('M').last() / 100 / 12
    )
    rf[ccy] = r

rf = pd.concat(rf, axis=1).sort_index()


# 5. Part 3 (a) — convert local equity returns into USD

usd_ret = pd.DataFrame(index=local_ret.index)

for country in local_ret.columns:
    ccy = COUNTRY_CCY[country]
    r_loc = local_ret[country]
    spot = fx[ccy].reindex(r_loc.index.union(fx.index)).ffill()
    fx_r = spot.shift(-1) / spot - 1
    usd_ret[country] = (1 + r_loc) * (1 + fx_r) - 1

# 6. Part 3 (b) — subtract currency excess return X_i,t

X = {}

for ccy in COUNTRY_CCY.values():
    spot = fx[ccy].reindex(usd_ret.index.union(fx.index)).ffill()
    rf_i = rf[ccy].reindex(spot.index).ffill()
    rf_us = rf['USD'].reindex(spot.index).ffill()

    x_series = (spot.shift(-1) / spot) * (1 + rf_i) - (1 + rf_us)
    x_series = x_series[~x_series.index.duplicated(keep='last')]  # deduplicate

    X[ccy] = x_series

X = pd.DataFrame(X).reindex(usd_ret.index)

hedged_ret = usd_ret.copy()
for country in hedged_ret.columns:
    ccy = COUNTRY_CCY[country]
    hedged_ret[country] = usd_ret[country] - X[ccy]


usd_ret.to_csv('index_returns_usd.csv')
hedged_ret.to_csv('index_returns_hedged.csv')




✅ Done! Files written:
   • index_returns_usd.csv
   • index_returns_hedged.csv



C:\Users\benka\AppData\Local\Temp\ipykernel_97032\870925742.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M').last()
C:\Users\benka\AppData\Local\Temp\ipykernel_97032\870925742.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M').last()
C:\Users\benka\AppData\Local\Temp\ipykernel_97032\870925742.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M').last()
C:\Users\benka\AppData\Local\Temp\ipykernel_97032\870925742.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M').last()
C:\Users\benka\AppData\Local\Temp\ipykernel_97032\870925742.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M').last()
C:\Users\benka\AppData\Local\Temp\ipykernel_97032\

In [ ]:

# 3 (c) — Sample mean and std deviation (annualized)


def annualized_stats(returns_df):
    mean = returns_df.mean() * 12
    std = returns_df.std() * (12**0.5)
    return pd.DataFrame({'Mean (ann)': mean, 'Std Dev (ann)': std})


usd_ret = pd.read_csv('index_returns_usd.csv', index_col=0, parse_dates=True)
hedged_ret = pd.read_csv('index_returns_hedged.csv', index_col=0, parse_dates=True)

stats_usd = annualized_stats(usd_ret)
stats_hedged = annualized_stats(hedged_ret)

# Combine for comparison
stats_combined = pd.concat(
    [stats_usd.add_suffix(' — Unhedged'), stats_hedged.add_suffix(' — Hedged')],
    axis=1
)

stats_combined.to_csv('summary_stats_returns.csv')



# 3 (d) — Correlation matrices


corr_usd = usd_ret.corr()
corr_hedged = hedged_ret.corr()

corr_usd.to_csv('correlation_unhedged.csv')
corr_hedged.to_csv('correlation_hedged.csv')




📊 Part 3(c): Annualized stats saved as 'summary_stats_returns.csv'
📈 Part 3(d): Correlation matrices saved as:
   • correlation_unhedged.csv
   • correlation_hedged.csv
